# HDBSCAN Extra Analysis

This notebook is for exploratory analysis on top of the HDBSCAN asset impact pipeline outputs.

It does not rebuild the pipeline tables. It assumes the HDBSCAN pipeline notebook has already been run and that the DuckDB tables exist.

In [87]:
import duckdb
import pandas as pd

from IPython.display import Markdown, display

DB_PATH = 'developer_project.duckdb'
con = duckdb.connect(DB_PATH)

CLUSTER_PROFILE_TABLE = 'cluster_profile_asset_base_v2'
CLUSTER_ASSET_PRIORITY_TABLE = 'cluster_asset_priority_by_cluster_v2'
CLUSTER_PERSONA_PROFILE_TABLE = 'cluster_persona_profile_v2'
CLUSTER_JOURNEY_PROFILE_TABLE = 'cluster_journey_profile_v2'
CLUSTER_EFFORT_PROFILE_TABLE = 'cluster_effort_profile_v2'
CLUSTER_TOP_PERSONA_SUMMARY_TABLE = 'cluster_top_persona_summary_v2'
CLUSTER_TOP_ASSET_SUMMARY_TABLE = 'cluster_top_asset_summary_v2'
CLUSTER_CORRELATION_SUMMARY_TABLE = 'cluster_correlation_summary_v2'
CLUSTER_GROUP_ROLLUP_TABLE = 'cluster_group_rollup_summary_v2'
CLUSTER_GROUP_ASSET_PROFILE_TABLE = 'cluster_group_asset_profile_v2'
CLUSTER_GROUP_COMPOSITION_TABLE = 'cluster_group_composition_summary_v2'
ASSET_AUDIENCE_TABLE = 'asset_audience_summary_v2'


In [88]:
required_tables = [
    CLUSTER_PROFILE_TABLE,
    CLUSTER_ASSET_PRIORITY_TABLE,
    CLUSTER_PERSONA_PROFILE_TABLE,
    CLUSTER_JOURNEY_PROFILE_TABLE,
    CLUSTER_EFFORT_PROFILE_TABLE,
    CLUSTER_TOP_PERSONA_SUMMARY_TABLE,
    CLUSTER_TOP_ASSET_SUMMARY_TABLE,
    CLUSTER_CORRELATION_SUMMARY_TABLE,
    CLUSTER_GROUP_ROLLUP_TABLE,
    CLUSTER_GROUP_ASSET_PROFILE_TABLE,
    CLUSTER_GROUP_COMPOSITION_TABLE,
    ASSET_AUDIENCE_TABLE,
]

missing = []
for table_name in required_tables:
    try:
        con.execute(f'SELECT 1 FROM {table_name} LIMIT 1')
    except Exception:
        missing.append(table_name)

if missing:
    raise ValueError(f'Missing pipeline output tables: {missing}')

display(Markdown('### Pipeline Tables Available'))
display(pd.DataFrame({'table_name': required_tables}))

### Pipeline Tables Available

,table_name
0,cluster_profile_asset_base_v2
1,cluster_asset_priority_by_cluster_v2
2,cluster_persona_profile_v2
3,cluster_journey_profile_v2
4,cluster_effort_profile_v2
5,cluster_top_persona_summary_v2
6,cluster_top_asset_summary_v2
7,cluster_correlation_summary_v2
8,cluster_group_rollup_summary_v2
9,cluster_group_asset_profile_v2


## Lifecycle Group Analysis

These views keep the analysis at the `active`, `cooling`, and `at_risk` level.

In [89]:
display(Markdown('### Lifecycle Group Rollup Summary'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_GROUP_ROLLUP_TABLE}
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
""").fetchdf())

display(Markdown('### Lifecycle Group Asset Profile'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_GROUP_ASSET_PROFILE_TABLE}
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
""").fetchdf())

display(Markdown('### Lifecycle Group Composition Table'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_GROUP_COMPOSITION_TABLE}
ORDER BY CASE cluster_group WHEN 'active' THEN 1 WHEN 'cooling' THEN 2 WHEN 'at_risk' THEN 3 ELSE 4 END
""").fetchdf())

### Lifecycle Group Rollup Summary

,cluster_group,developers,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share,top_effort_1,top_effort_1_share,top_effort_2,top_effort_2_share,top_volume_asset_1,top_volume_asset_2,top_volume_asset_3,top_breadth_asset_1,top_intensity_asset_1
0,active,418049,GenAI,0.698562,CUDA,0.138912,Robotics,0.060670,very high effort,0.962586,low effort,0.031159,ngc_download,devzone_download,forum_contribution,devzone_download,ngc_download
1,cooling,356500,GenAI,0.434056,CUDA,0.285316,Robotics,0.094003,very high effort,0.535352,high effort,0.390418,devzone_download,ngc_download,dli_training,devzone_download,ngc_download
2,at_risk,1580877,CUDA,0.408582,GenAI,0.340516,Robotics,0.082461,low effort,0.387062,medium effort,0.307890,devzone_download,ngc_download,dli_training,devzone_download,ngc_download


### Lifecycle Group Asset Profile

,cluster_group,developers,pct_dli_training,lift_dli_training,intensity_dli_training,pct_webinar,lift_webinar,intensity_webinar,pct_forum_contribution,lift_forum_contribution,intensity_forum_contribution,pct_bug_filed,lift_bug_filed,intensity_bug_filed,pct_hackathon,lift_hackathon,intensity_hackathon,pct_devzone_download,lift_devzone_download,intensity_devzone_download,pct_ngc_download,lift_ngc_download,intensity_ngc_download
0,active,418049,0.086629,0.733580,2.564904,0.016890,0.822280,2.366520,0.010542,1.264678,69.153846,0.001526,1.746666,30.040752,0.000033,0.268460,1.071429,0.174097,0.412078,40.791566,0.017670,1.314477,732.196426
1,cooling,356500,0.183868,1.557015,2.034570,0.042275,2.058084,1.847787,0.013966,1.675509,12.721028,0.001433,1.640506,19.906067,0.000070,0.562160,1.280000,0.339518,0.803620,25.668286,0.018463,1.373443,74.474020
2,at_risk,1580877,0.243432,2.061408,1.700735,0.037057,1.804042,1.609914,0.010389,1.246288,8.032698,0.001567,1.793988,20.974576,0.000083,0.669353,1.098485,0.409577,0.969447,14.468510,0.023156,1.722529,26.366361


### Lifecycle Group Composition Table

,cluster_group,developers,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share,top_journey_1,top_journey_1_share,top_journey_2,top_journey_2_share,top_effort_1,top_effort_1_share,top_effort_2,top_effort_2_share
0,active,418049,GenAI,0.698562,CUDA,0.138912,Robotics,0.060670,Evaluator,0.671924,Learner,0.204490,very high effort,0.962586,low effort,0.031159
1,cooling,356500,GenAI,0.434056,CUDA,0.285316,Robotics,0.094003,Historically_Active,0.929719,Builder,0.068519,very high effort,0.535352,high effort,0.390418
2,at_risk,1580877,CUDA,0.408582,GenAI,0.340516,Robotics,0.082461,Historically_Active,0.948792,Builder,0.048848,low effort,0.387062,medium effort,0.307890


## Cluster Analysis

These views stay at the per-cluster level so the lifecycle group summaries can be tied back to concrete cluster patterns.

In [90]:
display(Markdown('### Top Personas By Cluster'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_TOP_PERSONA_SUMMARY_TABLE}
ORDER BY cluster_label
""").fetchdf())

display(Markdown('### Top Assets By Cluster'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_TOP_ASSET_SUMMARY_TABLE}
ORDER BY cluster_label
""").fetchdf())

display(Markdown('### Cluster Correlation Summary'))
display(con.execute(f"""
SELECT *
FROM {CLUSTER_CORRELATION_SUMMARY_TABLE}
ORDER BY cluster_group, cluster_developers DESC, cluster_label
""").fetchdf())

### Top Personas By Cluster

,cluster_label,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share
0,Dormant_Former_Builders,CUDA,0.686126,Simulation,0.105529,Unknown,0.075067
1,Dormant_Low_Depth,CUDA,0.529681,GenAI,0.193532,Learning_Community,0.098306
2,Dormant_One_Time_Users,GenAI,0.371202,CUDA,0.230721,Unknown,0.192822
3,active_0,CUDA,0.583847,GenAI,0.153483,Unknown,0.115459
4,active_1,GenAI,0.957485,CUDA,0.018191,Unknown,0.016496
5,active_2,GenAI,0.460621,Learning_Community,0.197056,CUDA,0.167532
6,active_3,GenAI,0.975212,Unknown,0.022816,CUDA,0.000850
7,active_4,GenAI,0.832444,Robotics,0.081244,CUDA,0.059895
8,active_5,GenAI,0.845376,Unknown,0.104506,Robotics,0.019125
9,active_noise,CUDA,0.376407,GenAI,0.367827,Robotics,0.162012


### Top Assets By Cluster

,cluster_label,top_volume_asset_1,top_volume_asset_2,top_volume_asset_3,top_breadth_asset_1,top_breadth_asset_2,top_breadth_asset_3,top_intensity_asset_1,top_intensity_asset_2,top_intensity_asset_3
0,Dormant_Former_Builders,devzone_download,ngc_download,forum_contribution,devzone_download,ngc_download,dli_training,ngc_download,devzone_download,forum_contribution
1,Dormant_Low_Depth,devzone_download,dli_training,forum_contribution,devzone_download,dli_training,webinar,bug_filed,forum_contribution,ngc_download
2,Dormant_One_Time_Users,webinar,devzone_download,dli_training,webinar,devzone_download,dli_training,webinar,devzone_download,dli_training
3,active_0,devzone_download,ngc_download,webinar,devzone_download,ngc_download,webinar,ngc_download,devzone_download,webinar
4,active_1,ngc_download,webinar,devzone_download,ngc_download,webinar,devzone_download,ngc_download,webinar,devzone_download
5,active_2,dli_training,devzone_download,forum_contribution,dli_training,devzone_download,forum_contribution,bug_filed,forum_contribution,devzone_download
6,active_3,forum_contribution,ngc_download,bug_filed,forum_contribution,ngc_download,bug_filed,forum_contribution,ngc_download,bug_filed
7,active_4,devzone_download,webinar,forum_contribution,devzone_download,webinar,forum_contribution,devzone_download,webinar,forum_contribution
8,active_5,webinar,dli_training,devzone_download,webinar,dli_training,devzone_download,bug_filed,devzone_download,dli_training
9,active_noise,ngc_download,devzone_download,forum_contribution,devzone_download,dli_training,ngc_download,ngc_download,forum_contribution,devzone_download


### Cluster Correlation Summary

,cluster_label,cluster_group,cluster_developers,top_persona_1,top_persona_1_share,top_persona_2,top_persona_2_share,top_persona_3,top_persona_3_share,dominant_effort,dominant_effort_share,dominant_journey,dominant_journey_share,top_volume_asset_1,top_volume_asset_2,top_volume_asset_3,top_breadth_asset_1,top_breadth_asset_2,top_breadth_asset_3,top_intensity_asset_1,top_intensity_asset_2,top_intensity_asset_3,dominant_segment_persona,dominant_segment_effort,dominant_segment_journey,dominant_segment_developers,dominant_segment_share,dominant_segment_top_volume_asset,dominant_segment_top_breadth_asset,dominant_segment_top_intensity_asset
0,active_5,active,155034,GenAI,0.845376,Unknown,0.104506,Robotics,0.019125,very high effort,1.000000,Evaluator,0.984848,webinar,dli_training,devzone_download,webinar,dli_training,devzone_download,bug_filed,devzone_download,dli_training,GenAI,very high effort,Evaluator,130352,1.0,webinar,webinar,bug_filed
1,active_noise,active,105702,CUDA,0.376407,GenAI,0.367827,Robotics,0.162012,very high effort,0.852103,Learner,0.508259,ngc_download,devzone_download,forum_contribution,devzone_download,dli_training,ngc_download,ngc_download,forum_contribution,devzone_download,GenAI,very high effort,Learner,19543,1.0,dli_training,dli_training,devzone_download
2,active_1,active,66682,GenAI,0.957485,CUDA,0.018191,Unknown,0.016496,very high effort,0.999880,Evaluator,0.999475,ngc_download,webinar,devzone_download,ngc_download,webinar,devzone_download,ngc_download,webinar,devzone_download,GenAI,very high effort,Evaluator,63832,1.0,ngc_download,ngc_download,ngc_download
3,active_3,active,29409,GenAI,0.975212,Unknown,0.022816,CUDA,0.000850,very high effort,1.000000,Learner,0.999558,forum_contribution,ngc_download,bug_filed,forum_contribution,ngc_download,bug_filed,forum_contribution,ngc_download,bug_filed,GenAI,very high effort,Learner,28677,1.0,forum_contribution,forum_contribution,forum_contribution
4,active_2,active,24658,GenAI,0.460621,Learning_Community,0.197056,CUDA,0.167532,very high effort,1.000000,Evaluator,0.997769,dli_training,devzone_download,forum_contribution,dli_training,devzone_download,forum_contribution,bug_filed,forum_contribution,devzone_download,GenAI,very high effort,Evaluator,11349,1.0,dli_training,dli_training,bug_filed
5,active_4,active,18549,GenAI,0.832444,Robotics,0.081244,CUDA,0.059895,very high effort,1.000000,Evaluator,0.999030,devzone_download,webinar,forum_contribution,devzone_download,webinar,forum_contribution,devzone_download,webinar,forum_contribution,Robotics,very high effort,Evaluator,1507,1.0,forum_contribution,forum_contribution,forum_contribution
6,active_0,active,18015,CUDA,0.583847,GenAI,0.153483,Unknown,0.115459,very high effort,1.000000,Builder,1.000000,devzone_download,ngc_download,webinar,devzone_download,ngc_download,webinar,ngc_download,devzone_download,webinar,CUDA,very high effort,Builder,10518,1.0,devzone_download,devzone_download,ngc_download
7,at_risk_0,at_risk,436825,CUDA,0.387109,GenAI,0.342382,Robotics,0.096462,high effort,0.436747,Historically_Active,0.944740,devzone_download,dli_training,ngc_download,devzone_download,dli_training,webinar,ngc_download,devzone_download,forum_contribution,CUDA,high effort,Historically_Active,84553,1.0,devzone_download,devzone_download,forum_contribution
8,at_risk_5,at_risk,366741,CUDA,0.552003,GenAI,0.201047,Robotics,0.131842,medium effort,0.522562,Historically_Active,0.897025,devzone_download,dli_training,ngc_download,devzone_download,dli_training,webinar,devzone_download,ngc_download,forum_contribution,CUDA,medium effort,Historically_Active,113233,1.0,devzone_download,devzone_download,devzone_download
9,at_risk_2,at_risk,210376,GenAI,0.584515,CUDA,0.168674,Learning_Community,0.102726,medium effort,0.760168,Historically_Active,1.000000,dli_training,devzone_download,webinar,dli_training,devzone_download,webinar,devzone_download,dli_training,webinar,GenAI,medium effort,Historically_Active,110334,1.0,dli_training,dli_training,dli_train

In [91]:
display(Markdown('### Cluster Comparison View'))
display(con.execute(f"""
SELECT
    cluster_label,
    cluster_group,
    cluster_developers,
    top_persona_1,
    top_persona_1_share,
    dominant_journey,
    dominant_journey_share,
    dominant_effort,
    dominant_effort_share,
    top_volume_asset_1,
    top_volume_asset_2,
    top_breadth_asset_1,
    top_intensity_asset_1
FROM {CLUSTER_CORRELATION_SUMMARY_TABLE}
ORDER BY cluster_group, cluster_developers DESC, cluster_label
""").fetchdf())

### Cluster Comparison View

,cluster_label,cluster_group,cluster_developers,top_persona_1,top_persona_1_share,dominant_journey,dominant_journey_share,dominant_effort,dominant_effort_share,top_volume_asset_1,top_volume_asset_2,top_breadth_asset_1,top_intensity_asset_1
0,active_5,active,155034,GenAI,0.845376,Evaluator,0.984848,very high effort,1.000000,webinar,dli_training,webinar,bug_filed
1,active_noise,active,105702,CUDA,0.376407,Learner,0.508259,very high effort,0.852103,ngc_download,devzone_download,devzone_download,ngc_download
2,active_1,active,66682,GenAI,0.957485,Evaluator,0.999475,very high effort,0.999880,ngc_download,webinar,ngc_download,ngc_download
3,active_3,active,29409,GenAI,0.975212,Learner,0.999558,very high effort,1.000000,forum_contribution,ngc_download,forum_contribution,forum_contribution
4,active_2,active,24658,GenAI,0.460621,Evaluator,0.997769,very high effort,1.000000,dli_training,devzone_download,dli_training,bug_filed
5,active_4,active,18549,GenAI,0.832444,Evaluator,0.999030,very high effort,1.000000,devzone_download,webinar,devzone_download,devzone_download
6,active_0,active,18015,CUDA,0.583847,Builder,1.000000,very high effort,1.000000,devzone_download,ngc_download,devzone_download,ngc_download
7,at_risk_0,at_risk,436825,CUDA,0.387109,Historically_Active,0.944740,high effort,0.436747,devzone_download,dli_training,devzone_download,ngc_download
8,at_risk_5,at_risk,366741,CUDA,0.552003,Historically_Active,0.897025,medium effort,0.522562,devzone_download,dli_training,devzone_download,devzone_download
9,at_risk_2,at_risk,210376,GenAI,0.584515,Historically_Active,1.000000,medium effort,0.760168,dli_training,devzone_download,dli_training,devzone_download


## Asset-Centered Analysis

This flips the direction of the analysis from `group -> assets` to `asset -> audience`.

In [92]:
display(Markdown('### Asset Audience Table'))
display(con.execute(f"""
SELECT *
FROM {ASSET_AUDIENCE_TABLE}
ORDER BY exposed_developers DESC, asset_name
""").fetchdf())

display(Markdown('### Asset Audience: Sorted By Active Share'))
display(con.execute(f"""
SELECT *
FROM {ASSET_AUDIENCE_TABLE}
ORDER BY pct_active DESC, exposed_developers DESC, asset_name
""").fetchdf())

display(Markdown('### Asset Audience: Sorted By Learner Share'))
display(con.execute(f"""
SELECT *
FROM {ASSET_AUDIENCE_TABLE}
ORDER BY pct_learner DESC, exposed_developers DESC, asset_name
""").fetchdf())

### Asset Audience Table

,asset_name,asset_group,asset_role,exposed_developers,pct_active,pct_cooling,pct_at_risk,pct_builder,pct_learner,pct_high_effort,top_persona
0,devzone_download,activity_asset,download_activity,3962571,0.018367,0.030545,0.163402,0.077534,0.004258,0.248460,CUDA
1,dli_training,activity_asset,learning_activity,1107590,0.032697,0.059182,0.347453,0.029501,0.007366,0.404712,GenAI
2,webinar,activity_asset,learning_activity,192657,0.036651,0.078227,0.304074,0.072004,0.016449,0.441816,GenAI
3,ngc_download,activity_asset,download_activity,126082,0.058589,0.052204,0.290335,0.317857,0.006163,0.574539,GenAI
4,forum_contribution,activity_asset,community_activity,78181,0.056369,0.063686,0.210064,0.329978,0.014991,0.658894,CUDA
5,bug_filed,activity_asset,community_activity,8195,0.077852,0.062355,0.302379,0.259671,0.006711,0.738133,CUDA
6,hackathon,activity_asset,community_activity,1170,0.011966,0.021368,0.112821,0.128205,0.002564,0.271795,CUDA


### Asset Audience: Sorted By Active Share

,asset_name,asset_group,asset_role,exposed_developers,pct_active,pct_cooling,pct_at_risk,pct_builder,pct_learner,pct_high_effort,top_persona
0,bug_filed,activity_asset,community_activity,8195,0.077852,0.062355,0.302379,0.259671,0.006711,0.738133,CUDA
1,ngc_download,activity_asset,download_activity,126082,0.058589,0.052204,0.290335,0.317857,0.006163,0.574539,GenAI
2,forum_contribution,activity_asset,community_activity,78181,0.056369,0.063686,0.210064,0.329978,0.014991,0.658894,CUDA
3,webinar,activity_asset,learning_activity,192657,0.036651,0.078227,0.304074,0.072004,0.016449,0.441816,GenAI
4,dli_training,activity_asset,learning_activity,1107590,0.032697,0.059182,0.347453,0.029501,0.007366,0.404712,GenAI
5,devzone_download,activity_asset,download_activity,3962571,0.018367,0.030545,0.163402,0.077534,0.004258,0.248460,CUDA
6,hackathon,activity_asset,community_activity,1170,0.011966,0.021368,0.112821,0.128205,0.002564,0.271795,CUDA


### Asset Audience: Sorted By Learner Share

,asset_name,asset_group,asset_role,exposed_developers,pct_active,pct_cooling,pct_at_risk,pct_builder,pct_learner,pct_high_effort,top_persona
0,webinar,activity_asset,learning_activity,192657,0.036651,0.078227,0.304074,0.072004,0.016449,0.441816,GenAI
1,forum_contribution,activity_asset,community_activity,78181,0.056369,0.063686,0.210064,0.329978,0.014991,0.658894,CUDA
2,dli_training,activity_asset,learning_activity,1107590,0.032697,0.059182,0.347453,0.029501,0.007366,0.404712,GenAI
3,bug_filed,activity_asset,community_activity,8195,0.077852,0.062355,0.302379,0.259671,0.006711,0.738133,CUDA
4,ngc_download,activity_asset,download_activity,126082,0.058589,0.052204,0.290335,0.317857,0.006163,0.574539,GenAI
5,devzone_download,activity_asset,download_activity,3962571,0.018367,0.030545,0.163402,0.077534,0.004258,0.248460,CUDA
6,hackathon,activity_asset,community_activity,1170,0.011966,0.021368,0.112821,0.128205,0.002564,0.271795,CUDA


## Notes

- Keep adding exploratory tables here first.
- Once a table proves useful, decide later whether it belongs back in the pipeline.
- This notebook is the safer place for long-form interpretation and recommendation writing.

## Asset Impact by Lifecycle Group

This section applies the exposed-vs-unexposed and pre/post delta logic from `AssetImpact_Analysis_Runnable.ipynb` but groups by HDBSCAN lifecycle group (`active`, `cooling`, `at_risk`).

**What this answers:**
- Within each lifecycle group, do asset-exposed developers show stronger recent behavior than unexposed developers with the same baseline?
- Which asset exposures are associated with higher adoption outcomes (API usage, activity velocity)?
- Which asset combinations co-occur in each group?

**Caveats:**
- This is observational, not causal. Exposure and outcomes are measured from the same cross-sectional snapshot.
- Time windows (30-90d as baseline, 0-30d as recent) are a proxy for before/after, not true sequencing.
- `dev_profile_final_v4` has lifetime and windowed counts but no event-level timestamps.

### 0b. Raw Asset Exposure Rates by Cluster

What percentage of developers in each cluster have **ever** used each asset type? No normalization — just binary "touched it or not" rates. This answers the question the volume/breadth/intensity metrics obscure: how many developers in each cluster actually used each asset?

In [95]:
# ── Raw asset exposure rates: % of developers who ever used each asset ──

cluster_group_case = """
CASE
    WHEN cluster_label LIKE 'active_%' THEN 'active'
    WHEN cluster_label LIKE 'cooling_%' THEN 'cooling'
    WHEN cluster_label LIKE 'at_risk_%' THEN 'at_risk'
    ELSE NULL
END
"""

# ── By lifecycle group ──
display(Markdown('### Raw Exposure Rates by Lifecycle Group'))

group_exposure_sql = f"""
SELECT
    {cluster_group_case} AS cluster_group,
    COUNT(*) AS developers,
    SUM(CASE WHEN lifetime_dli_training_count > 0 THEN 1 ELSE 0 END) AS n_training,
    ROUND(AVG(CASE WHEN lifetime_dli_training_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_training,
    SUM(CASE WHEN lifetime_webinar_count > 0 THEN 1 ELSE 0 END) AS n_webinar,
    ROUND(AVG(CASE WHEN lifetime_webinar_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_webinar,
    SUM(CASE WHEN lifetime_devzone_download_count > 0 THEN 1 ELSE 0 END) AS n_devzone_dl,
    ROUND(AVG(CASE WHEN lifetime_devzone_download_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_devzone_dl,
    SUM(CASE WHEN lifetime_ngc_download_count > 0 THEN 1 ELSE 0 END) AS n_ngc_dl,
    ROUND(AVG(CASE WHEN lifetime_ngc_download_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_ngc_dl,
    SUM(CASE WHEN lifetime_forum_count > 0 THEN 1 ELSE 0 END) AS n_forum,
    ROUND(AVG(CASE WHEN lifetime_forum_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_forum,
    SUM(CASE WHEN lifetime_bug_count > 0 THEN 1 ELSE 0 END) AS n_bug,
    ROUND(AVG(CASE WHEN lifetime_bug_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_bug,
    SUM(CASE WHEN lifetime_hackathon_count > 0 THEN 1 ELSE 0 END) AS n_hackathon,
    ROUND(AVG(CASE WHEN lifetime_hackathon_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_hackathon
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1
ORDER BY 1
"""

group_exposure = con.execute(group_exposure_sql).fetchdf()
display(group_exposure)

# ── By individual cluster ──
display(Markdown('### Raw Exposure Rates by Cluster'))

cluster_exposure_sql = f"""
SELECT
    cluster_label,
    {cluster_group_case} AS cluster_group,
    COUNT(*) AS developers,
    ROUND(AVG(CASE WHEN lifetime_dli_training_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_training,
    ROUND(AVG(CASE WHEN lifetime_webinar_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_webinar,
    ROUND(AVG(CASE WHEN lifetime_devzone_download_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_devzone_dl,
    ROUND(AVG(CASE WHEN lifetime_ngc_download_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_ngc_dl,
    ROUND(AVG(CASE WHEN lifetime_forum_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_forum,
    ROUND(AVG(CASE WHEN lifetime_bug_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_bug,
    ROUND(AVG(CASE WHEN lifetime_hackathon_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_hackathon,
    ROUND(AVG(CASE WHEN activity_count_0_30d > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_recent_activity,
    ROUND(AVG(CASE WHEN high_effort_count_0_30d > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_recent_high_effort
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1, 2
ORDER BY cluster_group, developers DESC
"""

cluster_exposure = con.execute(cluster_exposure_sql).fetchdf()
display(cluster_exposure)

# ── Heatmap-style view: just the percentages pivoted ──
display(Markdown('### Exposure Rate Heatmap (percentages only)'))

pct_cols = ['pct_training', 'pct_webinar', 'pct_devzone_dl', 'pct_ngc_dl', 'pct_forum', 'pct_bug', 'pct_hackathon']
heatmap = cluster_exposure.set_index('cluster_label')[pct_cols].copy()
heatmap.columns = ['Training', 'Webinar', 'DevZone DL', 'NGC DL', 'Forum', 'Bug Filed', 'Hackathon']

styled = heatmap.style.background_gradient(cmap='YlOrRd', axis=None).format('{:.1%}')
display(styled)

### Raw Exposure Rates by Lifecycle Group

,cluster_group,developers,n_training,pct_training,n_webinar,pct_webinar,n_devzone_dl,pct_devzone_dl,n_ngc_dl,pct_ngc_dl,n_forum,pct_forum,n_bug,pct_bug,n_hackathon,pct_hackathon
0,active,418049,36215.0,0.0866,7061.0,0.0169,72781.0,0.1741,7387.0,0.0177,4407.0,0.0105,638.0,0.0015,14.0,0.0000
1,at_risk,1580877,384836.0,0.2434,58582.0,0.0371,647491.0,0.4096,36606.0,0.0232,16423.0,0.0104,2478.0,0.0016,132.0,0.0001
2,cooling,356500,65549.0,0.1839,15071.0,0.0423,121038.0,0.3395,6582.0,0.0185,4979.0,0.0140,511.0,0.0014,25.0,0.0001


### Raw Exposure Rates by Cluster

,cluster_label,cluster_group,developers,pct_training,pct_webinar,pct_devzone_dl,pct_ngc_dl,pct_forum,pct_bug,pct_hackathon,pct_recent_activity,pct_recent_high_effort
0,active_5,active,155034,0.0070,0.0098,0.0030,0.0000,0.0001,0.0001,0.0000,1.0,0.9848
1,active_noise,active,105702,0.1536,0.0506,0.4650,0.0694,0.0381,0.0058,0.0001,1.0,0.2577
2,active_1,active,66682,0.0000,0.0001,0.0000,0.0002,0.0000,0.0000,0.0000,1.0,0.9995
3,active_3,active,29409,0.0000,0.0000,0.0000,0.0001,0.0006,0.0000,0.0000,1.0,0.0004
4,active_2,active,24658,0.7660,0.0068,0.2169,0.0003,0.0136,0.0000,0.0000,1.0,0.9978
5,active_4,active,18549,0.0000,0.0004,0.0005,0.0000,0.0001,0.0000,0.0000,1.0,0.9990
6,active_0,active,18015,0.0001,0.0007,0.9887,0.0013,0.0001,0.0000,0.0001,1.0,0.9972
7,at_risk_0,at_risk,436825,0.2546,0.0435,0.4176,0.0084,0.0113,0.0005,0.0001,0.0,0.0000
8,at_risk_5,at_risk,366741,0.2026,0.0456,0.6691,0.0207,0.0172,0.0007,0.0001,0.0,0.0000
9,at_risk_2,at_risk,210376,0.7178,0.0101,0.1444,0.0000,0.0044,0.0000,0.0000,0.0,0.0000


### Exposure Rate Heatmap (percentages only)

,Training,Webinar,DevZone DL,NGC DL,Forum,Bug Filed,Hackathon
cluster_label,,,,,,,
active_5,0.7%,1.0%,0.3%,0.0%,0.0%,0.0%,0.0%
active_noise,15.4%,5.1%,46.5%,6.9%,3.8%,0.6%,0.0%
active_1,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
active_3,0.0%,0.0%,0.0%,0.0%,0.1%,0.0%,0.0%
active_2,76.6%,0.7%,21.7%,0.0%,1.4%,0.0%,0.0%
active_4,0.0%,0.0%,0.1%,0.0%,0.0%,0.0%,0.0%
active_0,0.0%,0.1%,98.9%,0.1%,0.0%,0.0%,0.0%
at_risk_0,25.5%,4.3%,41.8%,0.8%,1.1%,0.1%,0.0%
at_risk_5,20.3%,4.6%,66.9%,2.1%,1.7%,0.1%,0.0%


In [97]:
# ── API usage and journey signals per cluster ──

display(Markdown('### API Usage & Journey Signals by Cluster'))

api_cluster_sql = f"""
SELECT
    cluster_label,
    {cluster_group_case} AS cluster_group,
    COUNT(*) AS developers,
    ROUND(AVG(CASE WHEN lifetime_api_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_has_api,
    ROUND(AVG(lifetime_api_count), 2) AS avg_api_count,
    ROUND(AVG(lifetime_activity_count), 2) AS avg_total_activity,
    ROUND(AVG(lifetime_discover_count), 2) AS avg_discover,
    ROUND(AVG(lifetime_learn_count), 2) AS avg_learn,
    ROUND(AVG(lifetime_evaluate_count), 2) AS avg_evaluate,
    ROUND(AVG(lifetime_build_count), 2) AS avg_build,
    ROUND(AVG(lifetime_champion_count), 2) AS avg_champion,
    ROUND(AVG(lifetime_high_effort_count), 2) AS avg_high_effort
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1, 2
ORDER BY cluster_group, developers DESC
"""

api_cluster = con.execute(api_cluster_sql).fetchdf()
display(api_cluster)

# ── Same view at lifecycle group level ──
display(Markdown('### API Usage & Journey Signals by Lifecycle Group'))

api_group_sql = f"""
SELECT
    {cluster_group_case} AS cluster_group,
    COUNT(*) AS developers,
    ROUND(AVG(CASE WHEN lifetime_api_count > 0 THEN 1.0 ELSE 0.0 END), 4) AS pct_has_api,
    ROUND(AVG(lifetime_api_count), 2) AS avg_api_count,
    ROUND(AVG(lifetime_activity_count), 2) AS avg_total_activity,
    ROUND(AVG(lifetime_discover_count), 2) AS avg_discover,
    ROUND(AVG(lifetime_learn_count), 2) AS avg_learn,
    ROUND(AVG(lifetime_evaluate_count), 2) AS avg_evaluate,
    ROUND(AVG(lifetime_build_count), 2) AS avg_build,
    ROUND(AVG(lifetime_champion_count), 2) AS avg_champion,
    ROUND(AVG(lifetime_high_effort_count), 2) AS avg_high_effort
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1
ORDER BY 1
"""

api_group = con.execute(api_group_sql).fetchdf()
display(api_group)

### API Usage & Journey Signals by Cluster

,cluster_label,cluster_group,developers,pct_has_api,avg_api_count,avg_total_activity,avg_discover,avg_learn,avg_evaluate,avg_build,avg_champion,avg_high_effort
0,active_5,active,155034,0.0019,0.00,1.03,1.00,0.03,0.00,0.00,0.00,1.00
1,active_noise,active,105702,0.3146,30.85,116.74,33.74,1.66,5.56,69.62,6.16,46.82
2,active_1,active,66682,0.9581,11.77,12.85,12.84,0.00,0.00,0.00,0.00,1.01
3,active_3,active,29409,0.9991,37.05,38.05,38.05,0.00,0.00,0.00,0.00,1.00
4,active_2,active,24658,0.0008,0.00,2.20,1.06,0.89,0.23,0.00,0.02,1.20
5,active_4,active,18549,0.0007,0.00,1.04,1.04,0.00,0.00,0.00,0.00,1.00
6,active_0,active,18015,0.0026,0.00,2.31,1.10,0.00,0.07,1.15,0.00,1.02
7,at_risk_0,at_risk,436825,0.0212,0.07,9.66,1.45,0.60,1.36,6.13,0.12,1.16
8,at_risk_5,at_risk,366741,0.0489,0.31,17.49,1.97,1.53,2.57,11.32,0.10,1.11
9,at_risk_2,at_risk,210376,0.0136,0.01,2.00,1.06,0.80,0.14,0.00,0.00,1.00


### API Usage & Journey Signals by Lifecycle Group

,cluster_group,developers,pct_has_api,avg_api_count,avg_total_activity,avg_discover,avg_learn,avg_evaluate,avg_build,avg_champion,avg_high_effort
0,active,418049,0.3035,12.29,34.90,13.78,0.48,1.42,17.65,1.56,12.60
1,at_risk,1580877,0.0271,0.43,9.35,1.82,1.00,1.23,5.18,0.13,1.52
2,cooling,356500,0.0986,1.10,13.44,2.61,0.76,1.76,7.98,0.32,2.09


In [98]:
# Re-open connection if needed (safe to run if already open)
import duckdb, pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 100)

DB_PATH = 'developer_project.duckdb'
try:
    con.execute("SELECT 1")
except Exception:
    con = duckdb.connect(DB_PATH)

PROFILE_TABLE = 'cluster_profile_asset_base_v2'

# Confirm columns we need exist
schema = con.execute(f"DESCRIBE {PROFILE_TABLE}").fetchdf()
available_cols = set(schema['column_name'].astype(str))

has_api = 'lifetime_api_count' in available_cols
has_velocity = 'activity_velocity_0_30_vs_30_90' in available_cols

print(f"Table: {PROFILE_TABLE}")
print(f"Has lifetime_api_count: {has_api}")
print(f"Has activity_velocity: {has_velocity}")
print(f"Total columns: {len(available_cols)}")

Table: cluster_profile_asset_base_v2
Has lifetime_api_count: True
Has activity_velocity: True
Total columns: 148


### 1. Exposed vs Unexposed: Pre/Post Deltas by Lifecycle Group

For each asset type (training, webinar, download), compare exposed vs unexposed developers within each lifecycle group. Uses 30-90d as baseline and 0-30d as recent window.

In [99]:
# ── Cluster group case expression (reuse from pipeline) ──
cluster_group_case = """
CASE
    WHEN cluster_label LIKE 'active_%' THEN 'active'
    WHEN cluster_label LIKE 'cooling_%' THEN 'cooling'
    WHEN cluster_label LIKE 'at_risk_%' THEN 'at_risk'
    ELSE NULL
END
"""

api_avg = "AVG(lifetime_api_count) AS avg_lifetime_api_count," if has_api else ""
velocity_avg = "AVG(activity_velocity_0_30_vs_30_90) AS avg_activity_velocity," if has_velocity else ""

# ── Training: exposed vs unexposed by lifecycle group ──
display(Markdown('### Training Exposure: Deltas by Lifecycle Group'))

training_delta_sql = f"""
SELECT
    {cluster_group_case} AS cluster_group,
    CASE WHEN lifetime_dli_training_count > 0 THEN 'Training Exposed' ELSE 'Not Exposed' END AS exposure,
    COUNT(*) AS developers,
    AVG(activity_count_30_90d) AS avg_activity_30_90d,
    AVG(activity_count_0_30d) AS avg_activity_0_30d,
    AVG(activity_count_0_30d - activity_count_30_90d) AS delta_activity,
    AVG(high_effort_count_30_90d) AS avg_high_effort_30_90d,
    AVG(high_effort_count_0_30d) AS avg_high_effort_0_30d,
    AVG(high_effort_count_0_30d - high_effort_count_30_90d) AS delta_high_effort,
    {api_avg}
    {velocity_avg}
    AVG(weighted_recent_activity) AS avg_weighted_recent_activity
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 2
"""

training_delta = con.execute(training_delta_sql).fetchdf()
display(training_delta)

# ── Webinar: exposed vs unexposed by lifecycle group ──
display(Markdown('### Webinar Exposure: Deltas by Lifecycle Group'))

webinar_delta_sql = f"""
SELECT
    {cluster_group_case} AS cluster_group,
    CASE WHEN lifetime_webinar_count > 0 THEN 'Webinar Exposed' ELSE 'Not Exposed' END AS exposure,
    COUNT(*) AS developers,
    AVG(activity_count_30_90d) AS avg_activity_30_90d,
    AVG(activity_count_0_30d) AS avg_activity_0_30d,
    AVG(activity_count_0_30d - activity_count_30_90d) AS delta_activity,
    AVG(high_effort_count_30_90d) AS avg_high_effort_30_90d,
    AVG(high_effort_count_0_30d) AS avg_high_effort_0_30d,
    AVG(high_effort_count_0_30d - high_effort_count_30_90d) AS delta_high_effort,
    {api_avg}
    {velocity_avg}
    AVG(weighted_recent_activity) AS avg_weighted_recent_activity
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 2
"""

webinar_delta = con.execute(webinar_delta_sql).fetchdf()
display(webinar_delta)

# ── Download: exposed vs unexposed by lifecycle group ──
display(Markdown('### Download Exposure: Deltas by Lifecycle Group'))

download_delta_sql = f"""
SELECT
    {cluster_group_case} AS cluster_group,
    CASE WHEN (lifetime_devzone_download_count + lifetime_ngc_download_count) > 0
         THEN 'Download Exposed' ELSE 'Not Exposed' END AS exposure,
    COUNT(*) AS developers,
    AVG(activity_count_30_90d) AS avg_activity_30_90d,
    AVG(activity_count_0_30d) AS avg_activity_0_30d,
    AVG(activity_count_0_30d - activity_count_30_90d) AS delta_activity,
    AVG(high_effort_count_30_90d) AS avg_high_effort_30_90d,
    AVG(high_effort_count_0_30d) AS avg_high_effort_0_30d,
    AVG(high_effort_count_0_30d - high_effort_count_30_90d) AS delta_high_effort,
    {api_avg}
    {velocity_avg}
    AVG(weighted_recent_activity) AS avg_weighted_recent_activity
FROM {PROFILE_TABLE}
WHERE {cluster_group_case} IS NOT NULL
GROUP BY 1, 2
ORDER BY 1, 2
"""

download_delta = con.execute(download_delta_sql).fetchdf()
display(download_delta)

### Training Exposure: Deltas by Lifecycle Group

,cluster_group,exposure,developers,avg_activity_30_90d,avg_activity_0_30d,delta_activity,avg_high_effort_30_90d,avg_high_effort_0_30d,delta_high_effort,avg_lifetime_api_count,avg_activity_velocity,avg_weighted_recent_activity
0,active,Not Exposed,381834,5.477467,7.883379,2.405912,1.055286,1.302377,0.247092,12.696955,3.097577,6.754562
1,active,Training Exposed,36215,3.796631,5.079580,1.282949,0.836559,1.349330,0.512771,7.954328,1.868369,4.653798
2,at_risk,Not Exposed,1196041,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.516801,NaN,0.114240
3,at_risk,Training Exposed,384836,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.168038,NaN,0.090336
4,cooling,Not Exposed,290951,3.853979,0.000000,-3.853979,0.780833,0.000000,-0.780833,1.197233,0.000000,1.285557
5,cooling,Training Exposed,65549,3.100322,0.000000,-3.100322,0.953714,0.000000,-0.953714,0.686158,0.000000,1.050764


### Webinar Exposure: Deltas by Lifecycle Group

,cluster_group,exposure,developers,avg_activity_30_90d,avg_activity_0_30d,delta_activity,avg_high_effort_30_90d,avg_high_effort_0_30d,delta_high_effort,avg_lifetime_api_count,avg_activity_velocity,avg_weighted_recent_activity
0,active,Not Exposed,410988,5.304897,7.645369,2.340472,1.022015,1.281298,0.259283,12.240226,3.003284,6.558173
1,active,Webinar Exposed,7061,6.901147,7.356465,0.455318,1.869990,2.770146,0.900156,14.956663,2.026789,7.410877
2,at_risk,Not Exposed,1522295,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.433913,NaN,0.108424
3,at_risk,Webinar Exposed,58582,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.379622,NaN,0.108342
4,cooling,Not Exposed,341429,3.755249,0.000000,-3.755249,0.834674,0.000000,-0.834674,1.113874,0.000000,1.250598
5,cooling,Webinar Exposed,15071,2.812753,0.000000,-2.812753,0.312985,0.000000,-0.312985,0.862849,0.000000,1.056347


### Download Exposure: Deltas by Lifecycle Group

,cluster_group,exposure,developers,avg_activity_30_90d,avg_activity_0_30d,delta_activity,avg_high_effort_30_90d,avg_high_effort_0_30d,delta_high_effort,avg_lifetime_api_count,avg_activity_velocity,avg_weighted_recent_activity
0,active,Download Exposed,75373,9.787935,11.682419,1.894485,5.179454,3.428814,-1.750640,7.155016,2.635621,11.087733
1,active,Not Exposed,342676,4.351726,6.751450,2.399725,0.125042,0.839621,0.714579,13.414712,3.092325,5.579448
2,at_risk,Download Exposed,666321,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.151505,NaN,0.176551
3,at_risk,Not Exposed,914556,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.636190,NaN,0.058783
4,cooling,Download Exposed,123239,6.840343,0.000000,-6.840343,0.746744,0.000000,-0.746744,0.661000,0.000000,2.329367
5,cooling,Not Exposed,233261,2.064404,0.000000,-2.064404,0.847424,0.000000,-0.847424,1.336923,0.000000,0.668101


In [102]:
con.close()